In [1]:
from platform import python_version
print(python_version())

3.12.12


### A cross-scale multimodal framework identifies clinically actionable immunotherapy biomarkers in melanoma (study, forked)

Huoshen W, Yuan K, Xiong J, Lin Y, Yu W, Xie Y, Yuan Q, Zhang X, Dong C, Sun C, Yi S. A cross-scale multimodal framework identifies clinically actionable immunotherapy biomarkers in melanoma through bulk to single-cell and spatial transcriptomics integration. Hum Genomics. 2026 Mar 16;20(1):77. doi: 10.1186/s40246-026-00919-w. PMID: 41840725; PMCID: PMC13104449.

https://pubmed.ncbi.nlm.nih.gov/41840725/

In [ ]:
import os, sys, re, yaml
from pathlib import Path
from dotenv import load_dotenv
import time
import requests
import json

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

from scipy.stats import spearmanr

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/rna_velocity")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, str(ROOT_SRC))

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir, pdreadcsv, pdwritecsv

root_figure = create_dir(ROOT0, "pictures")
root_results = create_dir(ROOT0, "results")

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/rna_velocity
ROOT_SRC added: /home/flavio/uv/rna_velocity/src


### Start with GSE148190 (Apr 07, 2020)

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE148190

**title**: Single cell RNA and TCR sequencing of tumor-infiltrating lymphocytes from human melanoma

**reference**: Mahuron KM, Moreau JM, Glasgow JE, Boda DP, Pauli ML, Gouirand V, Panjabi L, Grewal R, Luber JM, Mathur AN, Feldman RM, Shifrut E, Mehta P, Lowe MM, Alvarado MD, Marson A, Singer M, Wells J, Jupp R, Daud AI, Rosenblum MD. Layilin augments integrin activation to promote antitumor immunity. J Exp Med. 2020 Sep 7;217(9):e20192080. doi: 10.1084/jem.20192080. PMID: 32539073; PMCID: PMC7478725.

**features**: human, melanoma, n_samples=14, and scRNA 10x 5' GEX + VDJ

**platform**: GPL24676	Illumina NovaSeq 6000 (Homo sapiens)

https://seqout.org/p/SRP255479

`COUNT_ROOT` is the directory that **holds the `count_<library>/` folders kb writes** — not the index, not the FASTQs. `load_qc.py` reads:

```python
d = os.path.join(COUNT_ROOT, f"count_{lib}", "counts_unfiltered")
```

so with `COUNT_ROOT=/data/k409` it expects:

```
/data/k409/count_tumor_primary/counts_unfiltered/cells_x_genes.{mature,nascent,ambiguous}.mtx
/data/k409/count_lymph_node/counts_unfiltered/...
/data/k409/count_blood_PBMC/counts_unfiltered/...
```

**In most cases you don't set it.** The default is `.`, and if you run everything from one working directory the counts land there and `load_qc.py` finds them.

You caught a real inconsistency though: `count_libs.sh` was writing `count_<lab>/` into the current directory while only `load_qc.py` honoured the variable, so the two could disagree. Both now use `COUNT_ROOT`, and `run_all.sh` exports it, so a single value controls write and read:

```bash
COUNT_ROOT=$WSuv/counts \
KBREF=$WSref/kb_nac_human \
FQDIR=$WSuv/fastq \
VELO_PY=$PWD/.venv-velo/bin/python bash $WSsrc/run_all.sh
```

Full set of paths the pipeline takes:

| variable | default | meaning |
|---|---|---|
| `KBREF` | `kbref` | dir with `index.idx`, `t2g.txt`, `cdna_t2c.txt`, `nascent_t2c.txt` → yours is `$WSref/kb_nac_human` |
| `FQDIR` | `fastq` | dir with `<SRR>_1.fastq.gz` / `_2.fastq.gz` |
| `COUNT_ROOT` | `.` | where `count_<library>/` is written and read |
| `TARGETS` | `$FQDIR/targets.tsv` | optional md5 manifest; skipped if absent |
| `AUTHOR_CELLS` | `geo_barcodes/author_cells.json` | output of `fetch_author_barcodes.py` |
| `THREADS` / `MEM` | 8 / 16G | passed to `kb count` |
| `VELO_PY` | `python` | the numpy<2 interpreter |

Everything downstream — `k409_*.h5ad`, the CSVs, the JSONs — is written to the current working directory, so run from wherever you want the analysis outputs to live; only the bulky count directories follow `COUNT_ROOT`.

- [count_libs.sh]({{artifact:2fd8092a-bec6-4e3b-ab95-2f04ba6a80e3}}) — v3, writes into `$COUNT_ROOT`
- [run_all.sh]({{artifact:eed991c3-d1b4-4e7a-9c12-321388f878c8}}) — v2, exports `COUNT_ROOT` to both stages

### Project: SRP255479

https://www.ncbi.nlm.nih.gov/Traces/study/?acc=SRP255479&o=acc_s%3Aa

### SRR11492025

https://www.ncbi.nlm.nih.gov/sra/?term=SRR11492025

> Sample: melanoma_K409_tumor_GEX - tumor_primary
> Instrument: Illumina NovaSeq 6000  
> Strategy: RNA-Seq  
> Source: TRANSCRIPTOMIC  
> Selection: cDNA  
> Layout: PAIRED  
> Construction protocol: Single cells were lysed in droplets using the 10X Chromium instrument. Reverse transcription was performed and libraries were constructed according to the 10X Chromium Single Cell 5' Gene Expression and V(D)J Profiling Solution kit.

4 runs:

Run	# of Spots	# of Bases	Size	Published  
SRR11492025	181,375,026	55.9G	19.3Gb	2020-04-16  
SRR11492026	184,516,567	56.8G	19.5Gb	2020-04-16  
SRR11492027	180,769,769	55.7G	19.3Gb	2020-04-16  
SRR11492028	179,867,174	55.4G	19.2Gb	2020-04-16  


### SRR11492017

https://www.ncbi.nlm.nih.gov/sra/?term=SRR11492017

> Sample: melanoma_K409_LN_GEX - lymph_node
> idem

4 runs:

Run	# of Spots	# of Bases	Size	Published  
SRR11492017	164,516,361	50.7G	17.5Gb	2020-04-16  
SRR11492018	167,671,130	51.6G	17.7Gb	2020-04-18  
SRR11492019	163,350,000	50.3G	17.3Gb	2020-04-16  
SRR11492020	162,465,397	50G	17.3Gb	2020-04-16  



### SRR11492013

https://www.ncbi.nlm.nih.gov/sra/?term=SRR11492013

> Sample: melanoma_K409_blood_GEX - blood PBMC
> idem

4 runs:

Run	# of Spots	# of Bases	Size	Published  
SRR11492013	185,992,761	57.3G	19.9Gb	2020-04-16  
SRR11492014	189,143,936	58.3G	20.1Gb	2020-04-16  
SRR11492015	184,534,285	56.8G	19.7Gb	2020-04-16  
SRR11492016	183,385,834	56.5G	19.7Gb	2020-04-16  

Those four accessions are four **sequencing lanes of the same GEM well** — one 10x channel, one cell-barcode set — not four samples. Same structure for the others: blood = SRR11492013–16, node = SRR11492017–20, tumour = SRR11492025–28, each lane 21–25 GB.

So running only SRR11492025 is right, and the depth numbers say the other three buy almost nothing:

| library | lane used | reads | reads/cell | mapped reads per UMI | saturation | modelled UMI gain from all 4 lanes |
|---|---|---|---|---|---|---|
| tumour | SRR11492025 | 181.4 M | 79,237 | 15.0 | **93.3%** | ~0.00% |
| lymph node | SRR11492017 | 164.5 M | 25,407 | 4.8 | 79.2% | 0.87% |
| blood | SRR11492013 | 186.0 M | 33,921 | 7.0 | 85.6% | 0.10% |

The tumour library is the most saturated of the three: 15 mapped reads collapse to each UMI, i.e. the library was sequenced roughly 4× deeper than needed for 2,289 cells. 10x's own guidance for 5′ GEX is 20–50k reads/cell; one lane gives 79k.

Two caveats on the gain column, since it's a model rather than a measurement: it assumes uniform molecule abundance (real libraries have heavy-tailed abundance, so the true gain is somewhat above zero), and "mapped reads" includes reads on background barcodes, which inflates reads-per-UMI a little. Percent-level, either way — and it adds no cells, since the cell set is fixed by the authors' barcode list. The completed analysis used exactly one lane per library, so what you ran matches the reported results.

If you do want the full depth, `kb count` takes lanes as consecutive R1/R2 pairs, and `count_libs.sh` now accepts a comma list:

```bash
bash $WSsrc/count_libs.sh \
  tumor_primary:SRR11492025,SRR11492026,SRR11492027,SRR11492028 \
  lymph_node:SRR11492017,SRR11492018 \
  blood_PBMC:SRR11492013
```

I dry-ran it against a stubbed `kb` to confirm the argument order comes out as `SRR11492025_1 SRR11492025_2 SRR11492026_1 SRR11492026_2 …` — pairs interleaved, which is what kallisto expects. Budget ~95 GB of FASTQ and ~45 min of counting per four-lane library.

The one place extra depth would genuinely matter is the unspliced layer: the node library sits at 79% saturation and has the lowest unspliced fraction (7.2%), so if any library is worth a second lane it's that one — though the constraint there is 5′ chemistry, not depth.

- [count_libs.sh]({{artifact:07289540-197d-4138-b62d-1699ca17278d}}) — v4, comma-separated lanes per library
- [lane_saturation.csv]({{artifact:227effd0-7ded-41d8-a16e-66f830be4322}}) — the table above

Your numbers cross-check cleanly against the run: SRA reports 164,516,361 spots for SRR11492017 and 185,992,761 for SRR11492013, and `kb count` logged `n_processed` = 164,516,361 and 185,992,761 — exact matches, so nothing was lost in transfer or truncated. The lanes are also near-identical in size (162–189 M reads, 50–58 GB each), which is what you'd expect from four lanes of one flow cell rather than four separate captures.

What the full depth would give, and what it would recover:

| library | 1 lane | all 4 lanes | reads/cell (1 → 4) | mapped reads per UMI | molecules unseen |
|---|---|---|---|---|---|
| lymph node | 164.5 M | 658.0 M | 25.4k → 101.6k | 4.8 | 0.82% → ~0% |
| blood | 186.0 M | 743.1 M | 33.9k → 135.5k | 7.0 | 0.09% → ~0% |
| tumour | 181.4 M | 725.5 M | 79.2k → 317.0k | 15.0 | ~0% → ~0% |

The "molecules unseen" column is the decisive one: at 4.8× coverage per UMI, Poisson says you've already observed all but 0.8% of the molecules in the node library — the worst of the three. Blood is at 0.09%, tumour effectively zero. So the twelve extra lanes would cost **287 GB of download instead of 72 GB**, plus four times the counting time, to recover under 1% more UMIs and not a single additional cell (the cell set comes from the authors' barcode lists).

One lane per library is the right call for all three. Keep in mind what the model assumes — uniform molecule abundance, so the real gain is a little higher than stated for lowly-expressed genes where a single extra read matters. But that doesn't change the decision, and it doesn't touch the actual binding constraint for velocity: the unspliced fraction (7.2% in the node, 12.1% in the tumour) is set by 5′ chemistry and intron capture, not by sequencing depth. Adding lanes raises reads per UMI; it does not create intronic reads that the chemistry didn't capture.

If you ever want to settle it empirically rather than by model, the cheap version is to count one library at 25%, 50% and 100% of a single lane (`seqtk sample` or just head -n on the FASTQ pair) and plot UMIs per cell against reads — three kb runs, ~15 minutes, and it gives you the actual saturation curve for this library rather than a Poisson approximation.

[lane_saturation.csv]({{artifact:dc83cd8b-93dd-4705-a39f-c45f81a296bd}}) — v2, now with the four-lane totals and per-UMI coverage alongside the earlier saturation figures.

### "full depth"

By "full depth" I mean **all the reads the authors deposited for that one library** — the four lanes summed — as opposed to the single lane you ran, which is a quarter of them.

The distinction that matters is between the *library* and its *sequencing*:

- **The library** is a single 10x channel: cells loaded once into one GEM well, one cDNA prep, one GSM accession. Its molecular content — which cells, which barcodes, how many distinct mRNA molecules per cell — was fixed at the bench and cannot change afterwards.
- **The sequencing** is how many times you read that fixed pool of molecules. The authors ran it across four lanes, so there are four SRR accessions carrying ~165 M reads each for the node. All four contain the same cell barcodes and the same UMIs; only the read count differs.

So for the lymph node:

- one lane = 164.5 M reads → 25.4k reads per cell
- "full depth" = 658.0 M reads → 101.6k reads per cell

Same 6,475 cells, same molecules, four times the reading.

This is why depth stops helping. The useful quantity is **mapped reads per UMI** — how many times, on average, each distinct molecule was sequenced. At one lane that's 4.8 for the node, 7.0 for blood, 15.0 for the tumour. Since each read lands on a molecule roughly at random, the fraction of molecules never sampled is about e^(−coverage): 0.82%, 0.09%, ~0%. Everything after that is re-reading molecules you have already counted; PCR duplicates of the same molecule collapse onto the same UMI and add nothing to the count matrix.

So "adding the other three lanes" does not mean more cells, more genes, or more molecules — it means reading the same ~26 M node molecules about 19 times each instead of 4.8 times, and recovering under 1% of molecules that a single lane missed.

Where full depth genuinely matters is a library that is *undersequenced* — say 1–2 reads per UMI, where a third of molecules are still unobserved and per-cell counts are visibly truncated. None of these three is in that regime, which is unsurprising given the authors sequenced each channel to 25–80k reads per cell against 10x's 20–50k guidance.

Confirmed on both counts that matter: the **lane-to-library mapping is right** — I re-pulled it from ENA just now, and the three experiments partition exactly as you wrote (SRX8067978 → SRR11492025–28 tumour, SRX8067976 → SRR11492017–20 node, SRX8067975 → SRR11492013–16 blood) — and the comma syntax is what `count_libs.sh` v4 expects, dry-tested to emit `R1 R2 R1 R2 …` in pair order.

Three things will bite you as written:

**1. It will silently skip everything you've already counted.** The guard is `[ -s "$COUNT_ROOT/count_<lab>/counts_unfiltered/cells_x_genes.mature.mtx" ]`, so any library you counted at one lane is reported as "already counted" and left alone. Send the full-depth run somewhere else, or move the old ones aside:

```bash
COUNT_ROOT=$WSuv/counts_4lane bash $WSsrc/count_libs.sh \
  tumor_primary:SRR11492025,SRR11492026,SRR11492027,SRR11492028 \
  lymph_node:SRR11492017,SRR11492018,SRR11492019,SRR11492020 \
  blood_PBMC:SRR11492013,SRR11492014,SRR11492015,SRR11492016
```

Keeping them side by side is worth it anyway — it's the one clean way to measure what the extra depth actually bought, rather than modelling it.

**2. 18 of the 24 files have no md5 entry.** Your `targets.tsv` only covers the six files from the one-lane run, and the script skips verification for filenames it can't find there. Given the corruption you already saw once (SRR11492017_2 arrived size-complete and md5-wrong), you want checking on all of them. I generated the full manifest from ENA — 24 files, 279 GB, four columns (filename, bytes, url, md5):

```bash
TARGETS=$WSsrc/targets_all_lanes.tsv COUNT_ROOT=$WSuv/counts_4lane \
  bash $WSsrc/count_libs.sh tumor_primary:SRR11492025,...
```

It doubles as the download list — the third column is the HTTPS URL for each file, so the range-fetch script takes it directly.

**3. You need to download 18 more files first.** 279 GB total against the 72 GB you have; at the ~9 MB/s you were getting with 8 parallel range requests that's around 6–7 hours. Counting then takes roughly 40–45 minutes per library instead of 11.

Per-library totals from ENA, so you can budget: tumour 94.9 GB, blood 97.2 GB, node 86.9 GB.

Nothing wrong with doing this — but the earlier arithmetic stands: at 4.8–15 mapped reads per UMI you are re-reading molecules you have already counted, and the expected return is under 1% more UMIs with no additional cells. If the point is to *verify* that rather than take my word for it, the cheap version is one library at full depth (the node, since it is the least saturated at 4.8 reads/UMI) compared against its one-lane count — 87 GB and 45 minutes instead of 279 GB and half a day.

[targets_all_lanes.tsv]({{artifact:681bc9fd-24cf-4ffb-a996-b385bc759612}}) — all 24 files with sizes, URLs and md5s, in the format `count_libs.sh` reads via `TARGETS`.

You have **two lanes per library**, not four — so the command I confirmed would abort with `missing fq/SRR11492027_1.fastq.gz` on the first library (the script checks each FASTQ exists before counting). What's on disk:

- tumour: SRR11492025 + SRR11492026
- node: SRR11492017 + SRR11492018
- blood: SRR11492013 + SRR11492014

That's exactly the 2-lane set from the original plan, before I trimmed to one lane to save download time. Matching command:

```bash
TARGETS=$WSsrc/targets_all_lanes.tsv COUNT_ROOT=$WSuv/counts_2lane \
bash $WSsrc/count_libs.sh \
  tumor_primary:SRR11492025,SRR11492026 \
  lymph_node:SRR11492017,SRR11492018 \
  blood_PBMC:SRR11492013,SRR11492014
```

`targets_all_lanes.tsv` covers all 24 files, so all 12 of yours get md5-verified. Use a fresh `COUNT_ROOT` if you already counted at one lane, or the script will skip the libraries it finds.

What the second lane changes, using the actual ENA read counts:

| library | 2-lane reads | reads/cell | reads per UMI (1 → 2 lanes) | molecules unseen |
|---|---|---|---|---|
| tumour | 365.9 M | 159,847 | 15.0 → 30.3 | ~0% → ~0% |
| lymph node | 332.2 M | 51,303 | 4.8 → 9.7 | 0.82% → 0.006% |
| blood | 375.1 M | 68,418 | 7.0 → 14.1 | 0.09% → 0.0001% |

Since the data is already downloaded, the only cost is compute — roughly 22 minutes per library instead of 11 — and the node is the one case where it does something measurable, closing that 0.8% gap. For the tumour you'd be reading each molecule 30 times over; harmless, just pointless.

My recommendation: run the two lanes. Not because the extra depth matters much, but because it makes all three libraries consistent at the depth you actually have on disk, and it removes "would more reads have changed the unspliced fraction?" as an open question — you'll have the numbers from both, since the one-lane counts still exist in the other `COUNT_ROOT`. Comparing `cell_calling.json` between the two runs answers it directly: if the unspliced fraction moves by more than a few tenths of a percent, depth mattered after all; if it doesn't, the 7.2% node figure is chemistry, as I claimed.

### run_all.sh

```Bash

SRC=/home/flavio/uv/rna_velocity/src/scripts
WSdwn=/home/flavio/uv/rna_velocity/download
WSref=$WSdwn/ref
KBREF=$WSref/kb_nac_human

WSuv=/home/flavio/uv/rna_velocity/download/GSE148190
FQDIR=$WSuv/fastq


echo "== 1/6 kb count (three libraries, ~11 min each at 8 threads) =="

# COUNT_ROOT=.
# export KBREF FQDIR COUNT_ROOT
# bash "$SRC/count_libs.sh" tumor_primary:SRR11492025 lymph_node:SRR11492017 blood_PBMC:SRR11492013

TARGETS=$SRC/targets_all_lanes.tsv
COUNT_ROOT=$WSuv/counts_2lane
export KBREF FQDIR COUNT_ROOT TARGETS

bash $WSsrc/count_libs.sh \
  tumor_primary:SRR11492025,SRR11492026 \
  lymph_node:SRR11492017,SRR11492018 \
  blood_PBMC:SRR11492013,SRR11492014
```

In [ ]:
# Cells are the authors' CellRanger-called barcodes, which keeps this analysis
# comparable with the published one. The log-log knee is computed as well, but only
# recorded for the barcode-rank figure -- it is stricter, and in this dataset every
# knee barcode is a subset of the author call.
if not os.path.exists(self.BARCODE_JSON):
    raise SystemExit(f"{self.BARCODE_JSON} not found -- run fetch_author_barcodes.py first")
author = {k: set(v) for k, v in json.load(open(self.BARCODE_JSON)).items()}
adatas, stats, ranks = {}, {}, {}
for lib in LIBS:
    A = load_one(lib)                      # loaded once; ranks cached for the figure
    tot = np.asarray(A.X.sum(1)).ravel()
    thr, idx, ranked = knee_threshold(tot)
    ranks[lib] = ranked
    keep = np.isin(A.obs["barcode"].values, list(author[lib]))
    S = A[keep].copy()
    layer_tot = sum(float(S.layers[k].sum()) for k in ("spliced", "unspliced", "ambiguous"))
    stats[lib] = {"barcodes_total": int(A.n_obs), "author_cells": len(author[lib]),
                "author_cells_found": int(keep.sum()), "knee_threshold": thr,
                "knee_cells": int((tot >= thr).sum()),
                "median_umi": float(np.median(np.asarray(S.X.sum(1)).ravel())),
                "reads_unspliced_frac": float(S.layers["unspliced"].sum() / layer_tot)}
    adatas[lib] = S
    print(lib, stats[lib])
adata = ad.concat(adatas, label=None, index_unique=None)
adata.var = adatas["tumor_primary"].var.loc[adata.var_names]
adata.write("k409_raw_called.h5ad")
json.dump(stats, open("cell_calling.json", "w"), indent=1)
np.savez("barcode_ranks.npz", **ranks)
print("combined:", adata.shape)
